# GeoGuard — PoC Notebook
Team: GeoGuard · Country: Yemen · Pilot: UAE

This submission runs from bundled inputs to a website-compatible result without credentials,
network queries, model downloads or manual widget clicks. Run every cell in order.

It recomputes NO₂ daily/monthly/period summaries from an actual Earth Engine export
and packages a previously executed, dated methane result. It does not rerun methane
inference. Use `Geoguard.ipynb` for new imagery and interactive model checks.

NO₂ columns and methane candidates remain separate. Neither proves overall air safety.

In [1]:
from pathlib import Path
import json, sys, math, csv
root = Path.cwd().resolve()
if not (root / 'geoguard').is_dir() and (root.parent / 'geoguard').is_dir():
    root = root.parent
assert (root / 'geoguard').is_dir(), 'Run from the cloned Geoguard repository.'
sys.path.insert(0, str(root))
from geoguard.no2 import unpack_stacked, summarize
from geoguard.core import SITES, public_methane, export_bundle
config = json.loads((root / 'examples/input.json').read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))

{
  "start": "2026-01-01",
  "end_exclusive": "2026-09-01",
  "radius_m": 5000,
  "collection": "COPERNICUS/S5P/NRTI/L3_NO2",
  "no2_source": "sources/no2_earthengine_export.geojson",
  "methane_source": "docs/data/demo.json",
  "mode": "recompute_saved_no2_and_replay_dated_methane"
}


## 1. Read the example input and calculate NO₂ history
The source contains daily spatial reductions for two 5 km circles, January–August 2026.
Missing days stay unknown. Negative valid retrievals are retained. Each usable day
has equal weight in the period mean. Units are mol/m² through the atmospheric column,
not ground-level concentration or AQI.

In [2]:
raw = json.loads((root / config['no2_source']).read_text(encoding='utf-8'))
rows = unpack_stacked(raw['features'], config['start'], config['end_exclusive'])
no2 = summarize(rows, SITES, config['start'], config['end_exclusive'],
                config['radius_m'], config['collection'])
for site in no2['sites']:
    print(f"{site['name']}: {site['mean_mol_m2']:.12g} mol/m²; "
          f"{site['valid_days']}/{site['total_days']} usable days")

Dubai comparison area: 0.000207851401957 mol/m²; 226/243 usable days
Jebel Ali comparison area: 0.000206665923533 mol/m²; 224/243 usable days


## 2. Attach the existing methane evidence
This is a replay of the UAE check observed on 22 September 2026 against 9 September,
at 23.86479° N, 53.61893° E. It is a different location and date from the NO₂ study.
Zero candidate pixels does not mean zero methane. 96.04% is usable coverage, not accuracy.
No new model inference or forecast is claimed.

In [3]:
saved = json.loads((root / config['methane_source']).read_text(encoding='utf-8'))
methane = public_methane(saved['methane'])
print(json.dumps(methane, indent=2, ensure_ascii=False))

{
  "status": "no_candidate",
  "message": "No methane-like candidate passed the rule in the saved 22 September 2026 image. This is not a clean-air conclusion.",
  "checked_at": "2026-09-22T22:47:59.698378+00:00",
  "current_acquired_at": "2026-09-22T07:12:57.703362Z",
  "background_acquired_at": "2026-09-09T07:13:12.928948Z",
  "current_product_id": "S2B_MSIL1C_20260922T065619_N0513_R063_T39QYG_20260922T104758",
  "background_product_id": "S2A_MSIL1C_20260909T070321_N0512_R063_T39QYG_20260909T104214",
  "provider_available_at": "2026-09-22T11:37:44.508375Z",
  "latitude": 23.86479,
  "longitude": 53.61893,
  "requested_bounds": [
    53.609112391785885,
    23.85576116054776,
    53.62874760821411,
    23.87381882886855
  ],
  "pipeline_version": "asap-s2-v2",
  "model_id": "MARS-S2L/MARSS2L_20250326",
  "weights_sha256": "be634fb9e24dc4877f44c1ff9f69972e6f0453e30d70c0dc03677876340ef246",
  "candidate_pixels": 0,
  "common_valid_fraction": 0.9604,
  "history_count": 1,
  "forecast": {

## 3. Verify the numerical result against the committed example
The expected file is supplied for comparison. The calculations above use the raw
GeoJSON, not the expected means. Checks cover daily and monthly parity, missing-day
counts, and the location/date of the methane record.

In [4]:
expected = json.loads((root / 'examples/expected_summary.json').read_text(encoding='utf-8'))
assert no2['sites'] == saved['no2']['sites'], 'Daily/monthly values differ from archived evidence'
for site, reference in zip(no2['sites'], expected['no2']):
    assert site['id'] == reference['id']
    assert math.isclose(site['mean_mol_m2'], reference['mean_mol_m2'], rel_tol=1e-12)
    assert site['valid_days'] == reference['valid_days']
    assert site['total_days'] == reference['total_days']
for key, value in expected['methane'].items():
    assert methane[key] == value, key
print('Example checks passed.')

Example checks passed.


## 4. Export the output
`outputs/submission/result.json` can be imported into the demo website.
The CSV provides a compact table. Generation timestamps change each run;
the source observation dates and numerical results do not.

In [5]:
output = root / 'outputs/submission'
output.mkdir(parents=True, exist_ok=True)
result = export_bundle(output / 'result.json', no2, methane)
with (output / 'no2_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['id', 'mean_mol_m2', 'valid_days', 'total_days'])
    writer.writeheader()
    writer.writerows({key: site[key] for key in writer.fieldnames} for site in no2['sites'])
assert json.loads(result.read_text(encoding='utf-8'))['schema'] == 'geoguard-demo-v1'
print('Complete: outputs/submission/result.json and no2_summary.csv')

Complete: outputs/submission/result.json and no2_summary.csv


## Interpretation and next steps
The two close NO₂ averages alone do not establish a significant difference or a source.
Clouds, observation dates and retrieval uncertainty affect comparisons. The saved methane
record is evidence about one screened observation, not an air-quality verdict.

For fresh queries, open `Geoguard.ipynb`: Earth Engine sign-in is required for new NO₂
queries; the experimental methane workflow downloads public imagery and pretrained weights.
See `README.md`, `VALIDATION.md` and `THIRD_PARTY_NOTICES.md` for scope and provenance.